# Word embeddings

In this notebook, we will build our own simplified versions of common functions in order to replicate some key results. We will use pre-trained embeddings, but e.g. the library `gensim` also allows training own embeddings on a custom corpus.

### 1. Loading pre-trained embeddings

The pre-trained GloVe embeddings can be downloaded [here](https://nlp.stanford.edu/projects/glove/). This notebook uses the `Wikipedia 2024 + Gigaword 5` embeddings with 100 dimensions as they allow for faster computations. Yet, the higher dimensional embeddings on average yield better results. Note that some of the outcomes here will change when using embeddings based on different texts or with different dimensions.

In [ ]:
import numpy as np
import pandas as pd
import csv

# Read in the embeddings text file
embeddings = pd.read_csv(
    "wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt",
    sep=" ",
    header=None,
    encoding="utf-8",
    quoting=csv.QUOTE_NONE,
)

# Set column names
embeddings.columns = ["word"] + [f"dim_{i}" for i in range(embeddings.shape[1] - 1)]

# Set words as the index
embeddings = embeddings.set_index("word")

# Normalise all vectors to unit length
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = embeddings / norms

print(
    f"Loaded {len(embeddings)} words, each represented as a {embeddings.shape[1]} dimensional vector."
)

embeddings.head()

### 2. Functions

In [ ]:
def similar(word, top_n=5, embeddings=embeddings):
    """
    Finds words that are most similar to a given word using a collection
    of word embeddings.

    Parameters
    ----------
    word : str
        The target word to be compared to all other words.
    top_n : int
        The number of similar words to return.
    embeddings : pd.DataFrame
        A DataFrame of word embeddings normalised to unit length,
        where the index contains the word strings (one word per row)
        and each column corresponds to a single embedding dimension.

    Returns
    -------
    list of str
        The top_n most similar words, excluding the target word itself.
    """
    similarities = embeddings @ embeddings.loc[word]
    similarities = similarities.sort_values(ascending=False)
    return similarities.index[1 : top_n + 1].tolist()


def different(words, embeddings=embeddings):
    """
    Takes a set of words, computes the mean embedding vector, and determines
    which word's embedding is the least similar to this mean vector.

    Parameters
    ----------
    words : list of str
        A list of words for which the outlier is to be determined.
    embeddings : pd.DataFrame
        A DataFrame of word embeddings normalised to unit length,
        where the index contains the word strings (one word per row)
        and each column corresponds to a single embedding dimension.

    Returns
    -------
    str
        The word that is furthest from the mean embedding of the group.
    """
    word_vectors = embeddings.loc[words]
    mean_vector = word_vectors.mean(axis=0)
    similarities = word_vectors @ mean_vector
    return similarities.idxmin()


def analogies(a, b, c, top_n=1, embeddings=embeddings):
    """
    Computes analogies of the form a:b :: c:d using word embeddings.
    For example, man (a) is to woman (b) what king (c) is to ... (d).

    Parameters
    ----------
    a, b, c : str
        The three words defining the analogy.
    top_n : int
        The number of analogy completions to return.
    embeddings : pd.DataFrame
        A DataFrame of word embeddings normalised to unit length,
        where the index contains the word strings (one word per row)
        and each column corresponds to a single embedding dimension.

    Returns
    -------
    list of str
        The top_n analogy completions, excluding the original words a, b, c.
    """
    target_vector = embeddings.loc[c] - embeddings.loc[a] + embeddings.loc[b]
    similarities = embeddings @ target_vector
    similarities = similarities.sort_values(ascending=False)
    mask = ~similarities.index.isin([a, b, c])
    return similarities[mask].index[:top_n].tolist()

### 3. Illustrations

For a niche term, can we still find sensible similar ones?

In [ ]:
similar("pterosaur")

Who was not a physicist?

In [ ]:
different(["einstein", "bohr", "feynman", "mozart"])

Making it more difficult:

In [ ]:
different(["einstein", "bohr", "feynman", "fleming"])

Who was not on the moon?

In [ ]:
different(["armstrong", "aldrin", "einstein"])

Who was not a writer?

In [ ]:
different(["austen", "armstrong", "shakespeare"])

Maybe most surprisingly of all, the model also understands a wide range of analogies:

In [ ]:
analogies(a="einstein", b="scientist", c="picasso")

In [ ]:
analogies(a="david", b="goliath", c="small")

Adjectives:

In [ ]:
analogies(a="tall", b="taller", c="smart")

In [ ]:
analogies(a="fast", b="faster", c="good")

Adverbs:

In [ ]:
analogies(a="slow", b="slowly", c="careful")

In [ ]:
analogies(a="tall", b="tallest", c="quick")

Past tense:

In [ ]:
analogies(a="swim", b="swam", c="walk")

Capitals

In [ ]:
analogies(a="france", b="paris", c="peru")

In [ ]:
analogies(a="scotland", b="edinburgh", c="japan")

Literature:

In [ ]:
analogies(a="england", b="shakespeare", c="germany")

In [ ]:
analogies(a="england", b="shakespeare", c="spain")

Sports:

In [ ]:
analogies(a="arsenal", b="football", c="lakers")

Lastly, note that these examples are of course cherry-picked. There are many analogies for which the word-embedding models return unreasonable results, particularly when the dimensionality of embeddings is low. Nevertheless, the ability of a computer to perform such difficult tasks using the geometry of a word vector space is quite astonishing.

### 4. Visualisations

Next, let us see whether we can replicate common plots of word embeddings. For faster processing, we fit t-SNE and PCA only to a small subset of words rather than the entire vocabulary. Although the results are not as good as those obtained using the full vocabulary, the computations are much faster, and we can still observe interesting patterns.

__t-SNE__

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

exemplary_words = [
    "one",
    "two",
    "three",
    "four",
    "five",
    "football",
    "basketball",
    "baseball",
    "inflation",
    "recession",
    "economics",
]
exemplary_vectors = embeddings.loc[exemplary_words]

# t-SNE (perplexity should be < n_samples / 3)
tsne = TSNE(n_components=2, perplexity=2, random_state=42)
tsne_result = tsne.fit_transform(exemplary_vectors)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(tsne_result[:, 0], tsne_result[:, 1], s=10)
for i, word in enumerate(exemplary_words):
    ax.annotate(
        word,
        (tsne_result[i, 0], tsne_result[i, 1]),
        textcoords="offset points",
        xytext=(5, -10),
        fontsize=9,
    )
ax.set_title("t-SNE of Exemplary Word Embeddings")
plt.tight_layout()
plt.show()

While t-SNE is a useful tool for obtaining visual intuition about clusters in high-dimensional space, it is a nonlinear mapping that projects data from high-dimensional space into fewer dimensions while attempting to preserve cluster structure. However, distances _between_ clusters in the 2D plot may not accurately reflect the distances in the original high-dimensional space. In other words, t-SNE is designed mainly to preserve local neighborhoods, not global distances. We should therefore also not expect vectors such as man : woman :: king : queen to have an equally accurate parallelogram structure in the 2D plot as in high-dimensional space. Still, with some perplexity values, it works reasonably well for the example shown here.

In [ ]:
exemplary_words = ["man", "woman", "king", "queen"]
exemplary_vectors = embeddings.loc[exemplary_words]

tsne = TSNE(n_components=2, perplexity=2, random_state=42)
tsne_result = tsne.fit_transform(exemplary_vectors)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(tsne_result[:, 0], tsne_result[:, 1], s=10)
for i, word in enumerate(exemplary_words):
    ax.annotate(
        word,
        (tsne_result[i, 0], tsne_result[i, 1]),
        textcoords="offset points",
        xytext=(-30, 5),
        fontsize=9,
    )
ax.set_title("t-SNE: man, woman, king, queen")
plt.tight_layout()
plt.show()

__PCA__

Next, let us try PCA which reduces dimensions linearly.

In [ ]:
from sklearn.decomposition import PCA

exemplary_words = [
    "germany",
    "berlin",
    "france",
    "paris",
    "spain",
    "madrid",
    "china",
    "beijing",
    "vietnam",
    "hanoi",
]
exemplary_vectors = embeddings.loc[exemplary_words]

pca = PCA(n_components=2)
pca_result = pca.fit_transform(exemplary_vectors)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pca_result[:, 0], pca_result[:, 1], s=10)
for i, word in enumerate(exemplary_words):
    ax.annotate(
        word,
        (pca_result[i, 0], pca_result[i, 1]),
        textcoords="offset points",
        xytext=(-35, 5),
        fontsize=9,
    )
ax.set_title("PCA: Countries and Capitals")
plt.tight_layout()
plt.show()

The capital vectors are in similar directions from the country vectors also in 2D space.